# CeNN-X2 — GPU-first dual-timescale cellular mLSTM layer

**New research layer**, extending the previous CeNN-xLSTM GPU prototype with **fast+slow memory, two dilated depthwise cellular convolutions, a learned time-scale router, optional bounded SDPA, and causal / bidirectional modes**.

No claim of pretrained accuracy, end-to-end acceleration, or validated CUDA speed is made by this notebook. Start with **Runtime → Change runtime type → T4 GPU**. This notebook is self-contained and creates the source files locally.

In [ ]:
import torch, time, statistics, sys
from pathlib import Path
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else '(CPU only)')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Create the module locally (no GitHub checkout required)
The cells below contain the exact reference source files. They are not precompiled CUDA kernels.

In [ ]:
PROJECT = Path('/content/cenn_x2') if Path('/content').is_dir() else Path.cwd() / 'cenn_x2'
PACKAGE = PROJECT / 'tinycenn_lm'
PACKAGE.mkdir(exist_ok=True, parents=True)
(PACKAGE / '__init__.py').write_text('')
(PACKAGE / 'cenn_xlstm_gpu.py').write_text('"""Chunk-parallel xLSTM-inspired CeNN mixer (experimental).\n\nIndependent implementation. mLSTM-style stabilized scalar exponential gates,\nkey/value matrix memory, and a depthwise cellular convolution. The recurrence\nhas no tokenwise Python loop: one vectorized GEMM path per fixed-size chunk.\nThis is *not* a drop-in, pretrained xLSTM checkpoint or a fused Triton kernel.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass CeNNXLSTMConfig:\n    dim: int = 256\n    heads: int = 4\n    feature_dim: int = 32\n    value_dim: int = 64\n    chunk_size: int = 64\n    conv_kernel: int = 5\n    bidirectional: bool = False\n\n    def __post_init__(self):\n        if min(self.dim, self.heads, self.feature_dim, self.value_dim) <= 0:\n            raise ValueError("dim, heads and head dimensions must be positive")\n        if not 1 <= self.chunk_size <= 256:\n            raise ValueError("chunk_size must lie in [1, 256]")\n        if self.conv_kernel <= 0 or self.conv_kernel % 2 != 1:\n            raise ValueError("conv_kernel must be a positive odd number")\n\n\n@dataclass\nclass CeNNXLSTMState:\n    # All recurrent state tensors share [batch, heads] leading dimensions.\n    matrix: Tensor                 # [B,H,F,Dv], normalized matrix memory\n    normalizer: Tensor             # [B,H,F], normalized vector memory\n    log_scale: Tensor              # [B,H], log stabilization scale\n    conv_cache: Tensor | None = None  # [B,dim,K-1] in causal mode\n\n    @property\n    def nbytes(self) -> int:\n        return sum(t.numel() * t.element_size() for t in\n                   (self.matrix, self.normalizer, self.log_scale, self.conv_cache)\n                   if t is not None)\n\n\ndef initialize_state(batch: int, heads: int, features: int, values: int,\n                     *, device: torch.device, dtype: torch.dtype) -> CeNNXLSTMState:\n    return CeNNXLSTMState(\n        matrix=torch.zeros(batch, heads, features, values, device=device, dtype=dtype),\n        normalizer=torch.zeros(batch, heads, features, device=device, dtype=dtype),\n        log_scale=torch.zeros(batch, heads, device=device, dtype=dtype),\n    )\n\n\ndef chunkwise_cenn_mlstm(q: Tensor, k: Tensor, v: Tensor, log_forget: Tensor,\n                         log_input: Tensor, state: CeNNXLSTMState | None = None,\n                         *, chunk_size: int = 64) -> tuple[Tensor, CeNNXLSTMState]:\n    """Compute causal stabilized mLSTM memory with GEMMs and per-chunk scans.\n\n    q,k: [B,H,T,F], v: [B,H,T,Dv]; log_forget/log_input: [B,H,T].\n    m_t = max(log(f_t)+m_{t-1}, log(i_t)); C_t and n_t are scaled by exp(-m_t).\n    log_input can be -inf to skip padded positions; log_forget=0 then preserves state.\n    This reference constructs CxC matrices *per chunk*, never TxT globally.\n    State must be in the same dtype/device as inputs (use fp32 for numerical safety).\n    """\n    if not 1 <= chunk_size <= 256:\n        raise ValueError("chunk_size must lie in [1, 256]")\n    if q.ndim != 4 or k.shape != q.shape or v.shape[:3] != q.shape[:3]:\n        raise ValueError("expected q,k [B,H,T,F] and v [B,H,T,Dv]")\n    if q.shape[2] == 0 or log_forget.shape != q.shape[:3] or log_input.shape != q.shape[:3]:\n        raise ValueError("empty sequence or invalid gate shapes")\n    if any(x.dtype != q.dtype or x.device != q.device for x in (k, v, log_forget, log_input)):\n        raise ValueError("all inputs must have equal dtype and device")\n    b, h, t, f = q.shape\n    if state is None:\n        state = initialize_state(b, h, f, v.shape[-1], device=q.device, dtype=q.dtype)\n    c, n, m = state.matrix, state.normalizer, state.log_scale\n    if (c.shape != (b, h, f, v.shape[-1]) or n.shape != (b, h, f)\n            or m.shape != (b, h) or c.dtype != q.dtype or c.device != q.device):\n        raise ValueError("recurrent state mismatch")\n\n    outputs = []\n    for start in range(0, t, chunk_size):\n        stop = min(start + chunk_size, t)\n        qc, kc, vc = q[:, :, start:stop], k[:, :, start:stop], v[:, :, start:stop]\n        lf, li = log_forget[:, :, start:stop], log_input[:, :, start:stop]\n        p = lf.cumsum(dim=-1)                                      # [B,H,C]\n        # m_t = p_t + max(m_previous, max_{j<=t}(log(i_j) - p_j))\n        best = torch.maximum((li - p).cummax(dim=-1).values, m.unsqueeze(-1))\n        mt = p + best\n        old_scale = torch.exp(m.unsqueeze(-1) + p - mt)           # [B,H,C]\n\n        length = stop - start\n        lower = torch.ones(length, length, device=q.device, dtype=torch.bool).tril()\n        log_weights = (p.unsqueeze(-1) - p.unsqueeze(-2)\n                       + li.unsqueeze(-2) - mt.unsqueeze(-1))\n        weights = torch.exp(log_weights.clamp(max=0.0)).masked_fill(~lower, 0.0)\n        similarity = torch.matmul(qc, kc.transpose(-1, -2))\n        attention = similarity * weights\n        numerator = (torch.matmul(qc, c) * old_scale.unsqueeze(-1)\n                     + torch.matmul(attention, vc))\n        normalizer = (torch.sum(qc * n.unsqueeze(-2), dim=-1) * old_scale\n                      + attention.sum(dim=-1))\n        # Stabilized denominator from mLSTM. Clamping only guards exp overflow\n        # when a very long all-padding span is provided.\n        floor = torch.exp((-mt).clamp(max=30))\n        denominator = torch.maximum(normalizer.abs(), floor).clamp_min(1e-6)\n        outputs.append(numerator / denominator.unsqueeze(-1))\n\n        end_scale = old_scale[..., -1]\n        end_weights = torch.exp(\n            (p[..., -1:] - p + li - mt[..., -1:]).clamp(max=0.0)\n        )\n        c = c * end_scale[..., None, None] + torch.matmul(\n            kc.transpose(-1, -2), vc * end_weights.unsqueeze(-1)\n        )\n        n = n * end_scale[..., None] + (kc * end_weights.unsqueeze(-1)).sum(dim=-2)\n        m = mt[..., -1]\n    return torch.cat(outputs, dim=-2), CeNNXLSTMState(c, n, m, state.conv_cache)\n\n\ndef tokenwise_cenn_mlstm_reference(q: Tensor, k: Tensor, v: Tensor,\n                                   log_forget: Tensor, log_input: Tensor,\n                                   state: CeNNXLSTMState | None = None):\n    """Independent token-by-token numerical oracle, never use for fast inference."""\n    b, h, t, f = q.shape\n    if state is None:\n        state = initialize_state(b, h, f, v.shape[-1], device=q.device, dtype=q.dtype)\n    c, n, m = state.matrix, state.normalizer, state.log_scale\n    output = []\n    for i in range(t):\n        mm = torch.maximum(log_forget[:, :, i] + m, log_input[:, :, i])\n        forget = torch.exp(log_forget[:, :, i] + m - mm)\n        write = torch.exp(log_input[:, :, i] - mm)\n        c = (c * forget[..., None, None]\n             + k[:, :, i, :, None] * (v[:, :, i] * write[..., None])[:, :, None])\n        n = n * forget[..., None] + k[:, :, i] * write[..., None]\n        num = torch.einsum("bhf,bhfd->bhd", q[:, :, i], c)\n        denom_raw = torch.einsum("bhf,bhf->bh", q[:, :, i], n).abs()\n        denom = torch.maximum(denom_raw, torch.exp((-mm).clamp(max=30))).clamp_min(1e-6)\n        output.append(num / denom[..., None])\n        m = mm\n    return torch.stack(output, dim=-2), CeNNXLSTMState(c, n, m, state.conv_cache)\n\n\nclass GPUCeNNmLSTM(nn.Module):\n    """Depthwise cellular mixing + GPU-friendly matrix memory.\n\n    Causal mode supports bounded-state streaming with `return_state=True`.\n    Bidirectional mode runs two chunk scans and preserves encoder future context;\n    it does not support recurrent state API. Existing pretrained layers are NOT\n    numerically preserved on installation: distill and gate before replacement.\n    """\n\n    def __init__(self, config: CeNNXLSTMConfig):\n        super().__init__()\n        self.config = config\n        d, h, f, v = config.dim, config.heads, config.feature_dim, config.value_dim\n        self.norm = nn.LayerNorm(d)\n        self.cellular = nn.Conv1d(d, d, kernel_size=config.conv_kernel, groups=d, bias=True)\n        self.cellular_gain = nn.Parameter(torch.tensor(0.1))\n        self.qkv = nn.Linear(d, h * (f * 2 + v))\n        self.gates = nn.Linear(d, h * 2)\n        self.output = nn.Linear(h * v, d)\n        self.residual_gain = nn.Parameter(torch.tensor(-2.0))\n        nn.init.constant_(self.gates.bias[:h], 3.0)  # forget ≈ 0.95\n        nn.init.constant_(self.gates.bias[h:], -2.0) # restrained input updates\n\n    def _cellular(self, x: Tensor, state: CeNNXLSTMState | None):\n        d, kk = self.config.dim, self.config.conv_kernel - 1\n        xc = x.transpose(1, 2)\n        if self.config.bidirectional:\n            # Centered convolution for bidirectional ModernBERT-like encoders.\n            cells = self.cellular(F.pad(xc, (kk // 2, kk // 2)))\n            cache = None\n        else:\n            if state is None or state.conv_cache is None:\n                previous = xc.new_zeros(xc.shape[0], d, kk)\n            else:\n                previous = state.conv_cache\n                if previous.shape != (xc.shape[0], d, kk):\n                    raise ValueError("convolution cache shape mismatch")\n            cells = self.cellular(torch.cat([previous, xc], dim=-1))\n            cache = torch.cat([previous, xc], dim=-1)[:, :, -kk:].clone() if kk else xc.new_empty(xc.shape[0], d, 0)\n        return x + self.cellular_gain.tanh() * F.silu(cells.transpose(1, 2)), cache\n\n    def forward(self, x: Tensor, state: CeNNXLSTMState | None = None,\n                *, valid_mask: Tensor | None = None, return_state: bool = False):\n        if x.ndim != 3 or x.shape[-1] != self.config.dim or x.shape[1] < 1:\n            raise ValueError("expected x [B,T,dim] with T>0")\n        if self.config.bidirectional and (state is not None or return_state):\n            raise ValueError("bidirectional encoder mode has no streaming state API")\n        if valid_mask is not None and (valid_mask.shape != x.shape[:2] or valid_mask.dtype != torch.bool):\n            raise ValueError("valid_mask must be bool [B,T]")\n        source = self.norm(x)\n        if valid_mask is not None:\n            source = source.masked_fill(~valid_mask.unsqueeze(-1), 0)\n        features, cache = self._cellular(source, state)\n        b, t, _ = features.shape\n        h, f, dv = self.config.heads, self.config.feature_dim, self.config.value_dim\n        split = self.qkv(features).split((h*f, h*f, h*dv), dim=-1)\n        q = F.normalize(split[0].reshape(b, t, h, f), dim=-1).transpose(1, 2)\n        k = F.normalize(split[1].reshape(b, t, h, f), dim=-1).transpose(1, 2)\n        v = split[2].reshape(b, t, h, dv).transpose(1, 2)\n        gate_f, gate_i = self.gates(features).reshape(b, t, 2, h).unbind(dim=2)\n        log_f = F.logsigmoid(gate_f).transpose(1, 2)\n        log_i = gate_i.transpose(1, 2).clamp(-12, 8)\n        if valid_mask is not None:\n            pad = ~valid_mask[:, None]\n            log_f = log_f.masked_fill(pad, 0.0)\n            log_i = log_i.masked_fill(pad, float(\'-inf\'))\n        # Accumulate recurrent memory in FP32 for mixed-precision stability.\n        q, k, v, log_f, log_i = (z.float() for z in (q, k, v, log_f, log_i))\n        if state is not None:\n            recurrent = CeNNXLSTMState(state.matrix, state.normalizer, state.log_scale)\n        else:\n            recurrent = None\n        y, new_state = chunkwise_cenn_mlstm(q, k, v, log_f, log_i, recurrent,\n                                             chunk_size=self.config.chunk_size)\n        if self.config.bidirectional:\n            yr, _ = chunkwise_cenn_mlstm(q.flip(-2), k.flip(-2), v.flip(-2),\n                                         log_f.flip(-1), log_i.flip(-1),\n                                         chunk_size=self.config.chunk_size)\n            y = (y + yr.flip(-2)) * 0.5\n        merged = y.transpose(1, 2).reshape(b, t, h * dv)\n        output = x + torch.sigmoid(self.residual_gain) * self.output(merged.to(self.output.weight.dtype))\n        if valid_mask is not None:\n            output = output.masked_fill(~valid_mask.unsqueeze(-1), 0)\n        new_state.conv_cache = cache\n        return (output, new_state) if return_state else output\n\n    def recurrent_state_bytes(self, batch_size: int = 1) -> int:\n        c = self.config\n        recurrent = batch_size * c.heads * (c.feature_dim * c.value_dim + c.feature_dim + 1) * 4\n        conv = batch_size * c.dim * (c.conv_kernel - 1) * self.norm.weight.element_size()\n        return recurrent + conv\n')
(PACKAGE / 'cenn_x2_layer.py').write_text('"""CeNN-X2: multiscale cellular mixing + two-timescale chunkwise matrix memory.\n\nIndependent xLSTM-inspired experimental layer, not a pretrained xLSTM replica.\nOne batched recurrent memory call handles all timescales; causal mode supports\nstreaming and a bounded optional exact local SDPA branch. All accumulators use\nFP32. Training is parallel within chunks, with a Python loop *over chunks*.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\n\nfrom .cenn_xlstm_gpu import CeNNXLSTMState, chunkwise_cenn_mlstm\n\n\n@dataclass(frozen=True)\nclass CeNNX2Config:\n    dim: int = 256\n    heads: int = 4\n    key_dim: int = 32\n    value_dim: int = 64\n    chunk_size: int = 64\n    memory_scales: int = 2\n    conv_kernel: int = 5\n    conv_dilations: tuple[int, ...] = (1, 4)\n    local_window: int = 0\n    bidirectional: bool = False\n    residual_init: float = -3.0\n\n    def __post_init__(self):\n        if min(self.dim, self.heads, self.key_dim, self.value_dim, self.memory_scales) < 1:\n            raise ValueError(\'positive sizes required\')\n        if self.dim % self.heads:\n            raise ValueError(\'dim must be divisible by heads\')\n        if not 1 <= self.chunk_size <= 256:\n            raise ValueError(\'chunk_size must lie in [1,256]\')\n        if self.conv_kernel < 1 or self.conv_kernel % 2 == 0:\n            raise ValueError(\'conv_kernel must be positive odd\')\n        if not self.conv_dilations or any(d <= 0 for d in self.conv_dilations):\n            raise ValueError(\'conv_dilations must be positive, nonempty\')\n        if self.local_window < 0:\n            raise ValueError(\'local_window cannot be negative\')\n\n\n@dataclass\nclass CeNNX2State:\n    matrix: Tensor                    # [B, S*H, F, Dv], FP32\n    normalizer: Tensor                # [B, S*H, F], FP32\n    log_scale: Tensor                 # [B, S*H], FP32\n    conv_cache: Tensor                # [B, D, max_receptive_field - 1], input dtype\n    local_keys: Tensor | None = None  # [B,H,L,F], input dtype\n    local_values: Tensor | None = None # [B,H,L,Dv]\n    local_valid: Tensor | None = None # [B,L], bool\n\n    @property\n    def nbytes(self):\n        return sum(t.numel() * t.element_size() for t in (\n            self.matrix, self.normalizer, self.log_scale, self.conv_cache,\n            self.local_keys, self.local_values, self.local_valid\n        ) if t is not None)\n\n\ndef _safe_local_attention(q: Tensor, k: Tensor, v: Tensor, *, window: int,\n                          bidirectional: bool, chunk_size: int,\n                          valid: Tensor | None = None,\n                          past_keys: Tensor | None = None,\n                          past_values: Tensor | None = None,\n                          past_valid: Tensor | None = None\n                          ) -> tuple[Tensor, Tensor | None, Tensor | None, Tensor | None]:\n    """Bounded SDPA: query chunks, context <= C+2(W-1), no T-by-T matrix.\n\n    `window` is the number of causal keys, including the query. Bidirectional\n    mode attends up to window-1 neighbors on either side (including itself).\n    """\n    b, h, t, d = q.shape\n    if past_keys is not None:\n        if bidirectional:\n            raise ValueError(\'bidirectional mode cannot use a previous local cache\')\n        if past_values is None or past_valid is None:\n            raise ValueError(\'incomplete local cache\')\n        if (past_keys.shape[:2] != (b, h) or past_keys.shape[-1] != d\n            or past_values.shape[:3] != past_keys.shape[:3]\n            or past_valid.shape != (b, past_keys.shape[2])):\n            raise ValueError(\'bad local cache shape\')\n        cache_len = past_keys.shape[2]\n        keys = torch.cat((past_keys, k), dim=2)\n        values = torch.cat((past_values, v), dim=2)\n    else:\n        cache_len = 0\n        keys, values = k, v\n    all_valid = torch.ones(b, t, device=q.device, dtype=torch.bool) if valid is None else valid\n    if cache_len:\n        all_valid = torch.cat((past_valid, all_valid), dim=1)\n\n    result = []\n    for start in range(0, t, chunk_size):\n        stop = min(start + chunk_size, t)\n        begin_k = max(0, cache_len + start - window + 1)\n        end_k = min(keys.shape[2], cache_len + stop + (window - 1 if bidirectional else 0))\n        candidate = keys[:, :, begin_k:end_k]\n        cv = values[:, :, begin_k:end_k]\n        qpos = cache_len + torch.arange(start, stop, device=q.device)\n        kpos = torch.arange(begin_k, end_k, device=q.device)\n        offset = qpos[:, None] - kpos[None, :]\n        allowed = ((offset.abs() < window) if bidirectional\n                   else ((offset >= 0) & (offset < window)))\n        allowed = allowed[None, None] & all_valid[:, None, None, begin_k:end_k]\n        # Invalid *queries* can use their own key if valid is False; output is\n        # masked to zero by the caller, ensuring no all-false SDPA row NaNs.\n        # If every key is invalid (entire padding chunk), allow one dummy key.\n        fallback = ~allowed.any(dim=-1, keepdim=True)\n        # Never branch on a CUDA tensor: avoid a GPU->CPU synchronization.\n        one = (kpos[None, :] == begin_k)[None, None, :, :]\n        allowed = allowed | (fallback & one)\n        y = F.scaled_dot_product_attention(\n            q[:, :, start:stop], candidate, cv, attn_mask=allowed,\n            dropout_p=0.0, is_causal=False,\n        )\n        result.append(y)\n    if bidirectional:\n        return torch.cat(result, dim=2), None, None, None\n    keep = window - 1\n    if keep:\n        # clone prevents tiny cache views retaining an entire prefill allocation\n        return (torch.cat(result, dim=2), keys[:, :, -keep:].clone(),\n                values[:, :, -keep:].clone(), all_valid[:, -keep:].clone())\n    return (torch.cat(result, dim=2), keys[:, :, :0].clone(),\n            values[:, :, :0].clone(), all_valid[:, :0].clone())\n\n\nclass CeNNX2Layer(nn.Module):\n    """Hybrid CeNN xLSTM layer with exact chunk/streaming equivalence.\n\n    Memory matrix is scaled and normalized in FP32. Fast and slow gates use\n    *independent learned decay*, with shared Q/K/V and a learned per-token\n    router. Distinct dilation convolutions provide short/medium neighborhood\n    mixing, and optional local SDPA augments precise retrieval.\n\n    Output is residual-compatible [B,T,D]; it is NOT equivalent to a native\n    pretrained Transformer attention module without distillation and gating.\n    """\n    def __init__(self, config: CeNNX2Config):\n        super().__init__()\n        self.config = config\n        d, h, kd, vd, s = (config.dim, config.heads, config.key_dim,\n                           config.value_dim, config.memory_scales)\n        self.norm = nn.LayerNorm(d)\n        self.cellular = nn.ModuleList([\n            nn.Conv1d(d, d, config.conv_kernel, dilation=r, groups=d, bias=True)\n            for r in config.conv_dilations\n        ])\n        # Starts near identity but all branches learn, with a per-channel gain.\n        self.cellular_gain = nn.Parameter(torch.full((len(config.conv_dilations), d), -2.0))\n        self.qkv = nn.Linear(d, h * (2 * kd + vd))\n        self.forget = nn.Linear(d, h * s)\n        self.input_gate = nn.Linear(d, h)\n        self.time_router = nn.Linear(d, h * s)\n        self.local_router = nn.Linear(d, h) if config.local_window else None\n        self.output = nn.Linear(h * vd, d)\n        self.residual_gate = nn.Parameter(torch.full((d,), config.residual_init))\n        # Half-life separation at initialization: S=2 -> approx [0.86, 0.995].\n        values = torch.linspace(0.86, 0.995, s)\n        init = torch.logit(values).repeat_interleave(h)\n        with torch.no_grad():\n            self.forget.weight.zero_()\n            # memory-scale-major layout matches reshape [B,T,S,H].\n            self.forget.bias.copy_(init)\n            self.input_gate.bias.fill_(-2.0)\n            self.time_router.bias.zero_()\n            if self.local_router is not None:\n                self.local_router.bias.fill_(-2.0)\n\n    def _cache_length(self):\n        return (self.config.conv_kernel - 1) * max(self.config.conv_dilations)\n\n    def initial_state(self, batch: int, *, device, dtype) -> CeNNX2State:\n        c = self.config\n        sh = c.memory_scales * c.heads\n        return CeNNX2State(\n            torch.zeros(batch, sh, c.key_dim, c.value_dim, device=device, dtype=torch.float32),\n            torch.zeros(batch, sh, c.key_dim, device=device, dtype=torch.float32),\n            torch.zeros(batch, sh, device=device, dtype=torch.float32),\n            torch.zeros(batch, c.dim, self._cache_length(), device=device, dtype=dtype),\n        )\n\n    def _cellular(self, x: Tensor, state: CeNNX2State | None) -> tuple[Tensor, Tensor]:\n        xt = x.transpose(1, 2)\n        cache_length = self._cache_length()\n        if self.config.bidirectional:\n            joined = None\n            next_cache = xt.new_empty(x.shape[0], self.config.dim, 0)\n        else:\n            old = (torch.zeros(x.shape[0], self.config.dim, cache_length,\n                               device=x.device, dtype=x.dtype)\n                   if state is None else state.conv_cache)\n            if old.shape != (x.shape[0], self.config.dim, cache_length):\n                raise ValueError(\'bad convolution cache\')\n            joined = torch.cat((old, xt), dim=-1)\n            next_cache = joined[..., -cache_length:].clone() if cache_length else joined[..., :0].clone()\n        z = x\n        for i, (conv, dilation) in enumerate(zip(self.cellular, self.config.conv_dilations)):\n            if self.config.bidirectional:\n                padding = (self.config.conv_kernel - 1) * dilation // 2\n                spatial = conv(F.pad(xt, (padding, padding)))\n            else:\n                # All convolutions share one max-span cached prefix.\n                left = (self.config.conv_kernel - 1) * dilation\n                spatial = conv(joined[:, :, cache_length-left:])\n            z = z + torch.sigmoid(self.cellular_gain[i])[None, None] * F.silu(\n                spatial.transpose(1, 2)\n            )\n        return z, next_cache\n\n    def forward(self, x: Tensor, state: CeNNX2State | None = None,\n                *, valid_mask: Tensor | None = None, return_state: bool = False):\n        cfg = self.config\n        if x.ndim != 3 or x.shape[-1] != cfg.dim or x.shape[1] < 1:\n            raise ValueError(\'input must be [B,T,dim], T>=1\')\n        if cfg.bidirectional and (return_state or state is not None):\n            raise ValueError(\'bidirectional encoder has no streaming state\')\n        if valid_mask is not None and (valid_mask.shape != x.shape[:2] or valid_mask.dtype != torch.bool):\n            raise ValueError(\'valid_mask must have shape [B,T] and bool dtype\')\n        batch, t, _ = x.shape\n        if state is None:\n            state = self.initial_state(batch, device=x.device, dtype=x.dtype)\n        if (state.matrix.shape != (batch, cfg.heads * cfg.memory_scales, cfg.key_dim, cfg.value_dim)\n            or state.conv_cache.dtype != x.dtype or state.matrix.dtype != torch.float32\n            or state.matrix.device != x.device):\n            raise ValueError(\'invalid recurrent state\')\n        xn = self.norm(x)\n        if valid_mask is not None:\n            xn = xn.masked_fill(~valid_mask[..., None], 0)\n        feat, conv_cache = self._cellular(xn, state)\n        h, kd, vd, scales = cfg.heads, cfg.key_dim, cfg.value_dim, cfg.memory_scales\n        qq, kk, vv = self.qkv(feat).split((h*kd, h*kd, h*vd), dim=-1)\n        q = F.normalize(qq.reshape(batch, t, h, kd), dim=-1).transpose(1, 2)\n        k = F.normalize(kk.reshape(batch, t, h, kd), dim=-1).transpose(1, 2)\n        v = vv.reshape(batch, t, h, vd).transpose(1, 2)\n        forget = F.logsigmoid(self.forget(feat).reshape(batch, t, scales, h))\n        incoming = self.input_gate(feat).reshape(batch, t, h)\n        # Broadcast shared Q/K/V to scale-major, independently gated memory heads.\n        qrep = q[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, kd)\n        krep = k[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, kd)\n        vrep = v[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, vd)\n        log_f = forget.permute(0, 2, 3, 1).reshape(batch, scales*h, t)\n        log_i = incoming.transpose(1, 2)[:, None].expand(-1, scales, -1, -1).reshape(batch, scales*h, t)\n        if valid_mask is not None:\n            padding = ~valid_mask[:, None]\n            log_f = log_f.masked_fill(padding, 0.0)\n            log_i = log_i.masked_fill(padding, float(\'-inf\'))\n        in_state = CeNNXLSTMState(state.matrix, state.normalizer, state.log_scale)\n        y, new_mem = chunkwise_cenn_mlstm(qrep.float(), krep.float(), vrep.float(),\n                                           log_f.float(), log_i.float(), in_state,\n                                           chunk_size=cfg.chunk_size)\n        if cfg.bidirectional:\n            yr, _ = chunkwise_cenn_mlstm(\n                qrep.float().flip(2), krep.float().flip(2), vrep.float().flip(2),\n                log_f.float().flip(2), log_i.float().flip(2),\n                chunk_size=cfg.chunk_size)\n            y = 0.5 * (y + yr.flip(2))\n        # [B,S,H,T,V] -> [B,H,T,V] using query-conditioned time-scale routing.\n        ys = y.reshape(batch, scales, h, t, vd)\n        weights = self.time_router(feat).reshape(batch, t, scales, h)\n        weights = weights.softmax(dim=2).permute(0, 2, 3, 1)\n        merged = (ys * weights[..., None]).sum(dim=1)\n        if cfg.local_window:\n            local, new_keys, new_values, new_valid = _safe_local_attention(\n                q, k, v, window=cfg.local_window, bidirectional=cfg.bidirectional,\n                chunk_size=cfg.chunk_size, valid=valid_mask,\n                past_keys=(state.local_keys if not cfg.bidirectional else None),\n                past_values=(state.local_values if not cfg.bidirectional else None),\n                past_valid=(state.local_valid if not cfg.bidirectional else None),\n            )\n            gate = torch.sigmoid(self.local_router(feat)).transpose(1, 2).unsqueeze(-1)\n            merged = gate * local + (1 - gate) * merged.to(local.dtype)\n        else:\n            new_keys = new_values = new_valid = None\n        raw = merged.transpose(1, 2).reshape(batch, t, h * vd)\n        output = x + torch.sigmoid(self.residual_gate)[None, None] * self.output(\n            raw.to(self.output.weight.dtype)\n        )\n        if valid_mask is not None:\n            output = output.masked_fill(~valid_mask[..., None], 0)\n        next_state = CeNNX2State(new_mem.matrix, new_mem.normalizer,\n                                new_mem.log_scale, conv_cache, new_keys, new_values, new_valid)\n        return (output, next_state) if return_state else output\n\n    def state_bytes(self, batch: int = 1, dtype: torch.dtype = torch.float32) -> int:\n        c = self.config\n        count = batch * c.heads * c.memory_scales * (c.key_dim*c.value_dim + c.key_dim + 1)\n        n = count * 4 + batch*c.dim*self._cache_length()*torch.empty((), dtype=dtype).element_size()\n        if c.local_window:\n            n += batch*(c.local_window-1)*(c.heads*(c.key_dim+c.value_dim)*torch.empty((),dtype=dtype).element_size()+1)\n        return n\n')
sys.path.insert(0,str(PROJECT))
from tinycenn_lm.cenn_x2_layer import CeNNX2Config, CeNNX2Layer
print('Created:', PACKAGE)

## 2. Minimal forward pass and state size
Two memory timescales are batched together. All matrix-memory accumulators remain FP32 even with CUDA autocast.

In [ ]:
from tinycenn_lm.cenn_x2_layer import CeNNX2Config, CeNNX2Layer
model = CeNNX2Layer(CeNNX2Config(dim=256,heads=4,key_dim=32,value_dim=64,chunk_size=64,local_window=32)).to(DEVICE).eval()
x = torch.randn(2, 256 if DEVICE.type=='cuda' else 32, 256, device=DEVICE)
with torch.inference_mode():
    with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE.type=='cuda'):
        y, state = model(x, return_state=True)
print('input/output',tuple(x.shape),tuple(y.shape),'finite',bool(torch.isfinite(y).all()))
print('streaming state',state.nbytes,'bytes | model parameters',sum(p.numel() for p in model.parameters()))

## 3. Causal streaming equivalence
Check a whole sequence against awkwardly split pieces, including the local-attention cache.

In [ ]:
torch.manual_seed(11)
check = CeNNX2Layer(CeNNX2Config(dim=24,heads=3,key_dim=8,value_dim=8,chunk_size=7,conv_dilations=(1,3),local_window=5)).to(DEVICE).eval()
u = torch.randn(2, 29, 24,device=DEVICE)
with torch.inference_mode():
    full = check(u)
    p, state, begin = [], None, 0
    for end in [1,2,11,15,29]:
        z,state=check(u[:,begin:end],state,return_state=True)
        p.append(z); begin=end
    mixed=torch.cat(p,dim=1)
    delta=(full-mixed).abs().max().item()
    print('max streaming difference',delta)
    assert torch.allclose(full,mixed,atol=4e-5,rtol=4e-5)
print('PASS: full and streamed outputs agree')

## 4. Bidirectional encoder mode
Laya/ModernBERT is bidirectional; choose this for encoder experiments. This is not a drop-in pretrained ModernBERT attention replacement.

In [ ]:
enc=CeNNX2Layer(CeNNX2Config(dim=256,heads=4,key_dim=32,value_dim=64,chunk_size=64,bidirectional=True,local_window=32)).to(DEVICE)
with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'):
    z=enc(x)
print('bidirectional shape',tuple(z.shape),'finite',bool(torch.isfinite(z).all()))

## 5. CUDA/CPU inference microbenchmark
Timing compares *different operators* for diagnostics, not equal-quality models. A local SDPA comparison is not an end-to-end Transformer speed benchmark. CPU execution uses a deliberately small sequence to remain practical.

In [ ]:
from tinycenn_lm.cenn_xlstm_gpu import CeNNXLSTMConfig, GPUCeNNmLSTM
import torch.nn.functional as F
D,H,FEAT=256,4,32
prior=GPUCeNNmLSTM(CeNNXLSTMConfig(dim=D,heads=H,feature_dim=FEAT,value_dim=D//H,chunk_size=64)).to(DEVICE).eval()
new1=CeNNX2Layer(CeNNX2Config(dim=D,heads=H,key_dim=FEAT,value_dim=D//H,chunk_size=64,memory_scales=1,conv_dilations=(1,))).to(DEVICE).eval()
new2=CeNNX2Layer(CeNNX2Config(dim=D,heads=H,key_dim=FEAT,value_dim=D//H,chunk_size=64,memory_scales=2,conv_dilations=(1,4))).to(DEVICE).eval()
new_local=CeNNX2Layer(CeNNX2Config(dim=D,heads=H,key_dim=FEAT,value_dim=D//H,chunk_size=64,memory_scales=2,conv_dilations=(1,4),local_window=32)).to(DEVICE).eval()

def timed(module, inp, warmup=4, trials=10):
    results=[]
    with torch.inference_mode():
        for idx in range(warmup+trials):
            if DEVICE.type=='cuda':
                torch.cuda.synchronize();a=torch.cuda.Event(enable_timing=True);b=torch.cuda.Event(enable_timing=True);a.record()
                with torch.autocast('cuda',dtype=torch.float16): result=module(inp)
                b.record();torch.cuda.synchronize(); ms=a.elapsed_time(b)
            else:
                ts=time.perf_counter();result=module(inp);ms=(time.perf_counter()-ts)*1000
            if idx>=warmup: results.append(ms)
    return round(statistics.median(results),3),bool(torch.isfinite(result).all())

shapes=([128,512,1024] if DEVICE.type=='cuda' else [16])
for length in shapes:
    inp=torch.randn(2,length,D,device=DEVICE)
    print('\nLength',length,'batch=2')
    for name,fn in [('prior',prior),('X2-single',new1),('X2-dual',new2),('X2-dual-local',new_local)]:
        ms,finite=timed(fn,inp)
        print(f'  {name:15} {ms:9.3f} ms, finite={finite}')

## 6. Suggested first scientifically controlled experiment

1. Freeze the original **SmolLM2 layer 0** or **Laya layer 15/16** as teacher. Cache identical teacher input/output on disjoint train/probe/audit cases.
2. Compare CeNN-X2 single memory scale, dual scale, and dual+local branch at identical update budgets. Use native attention as the untouched baseline.
3. Select checkpoints only on development fidelity, NLL or task-level metrics; evaluate held-out performance once.
4. Benchmark complete model with paired CUDA events at multiple batch/context shapes. Preserve native layers when acceptance gates fail.

**No model publication until held-out quality and speed gates pass.**